# Event Streaming
- What you can stream
    - for event in stream: Raw protocol events with full envelope and access to every channel.
    - stream.messages: Model message streams, one per LLM call.
    - message.text: Text deltas and final text for a message.
    - message.reasoning: Reasoning deltas for models that expose reasoning content.
    - message.tool_calls: Tool-call argument chunks and finalized tool calls.
    - message.output: Final message object after the model call completes.
    - stream.values: Agent state snapshots.
    - stream.output: Final agent state.
    - stream.subgraphs: Nested graph runs (sub-agents and plain subgraphs).
    - stream.extensions: Custom transformer projections.
    - stream.tool_calls: Tool execution lifecycle, inputs, output deltas, final output, and errors.
- stream.messages yields ChatModelStream objects. Each message stream exposes .text, .reasoning, .tool_calls, and .output. 
- Sync projections are iterable for live deltas and drainable for final values: use str(message.text) for final text and message.tool_calls.get() for finalized tool calls.

In [ ]:
from langchain.agents import create_agent


def get_weather(city: str) -> str:
    """Get weather for a city."""
    return f"It's always sunny in {city}!"


agent = create_agent(
    model="gpt-5-nano",
    tools=[get_weather],
)

stream = agent.stream_events({
    "messages": [{"role": "user", "content": "What is the weather in SF?"}],
}, version="v3")

for message in stream.messages:
    for delta in message.text:
        print(delta, end="", flush=True)

final_state = stream.output

## Agent messages
message.output gives you the finalized AI message, including provider-specific content blocks.

In [ ]:
stream = agent.stream_events(input, version="v3")

for message in stream.messages:
    print(f"[{message.node}] ", end="")
    for delta in message.text:
        print(delta, end="", flush=True)

    full_message = message.output
    usage = full_message.usage_metadata
    if usage:
        print(usage)

## Reasoning content
Reasoning content uses the same shape as text content, but it is available only when the selected model emits reasoning blocks.

In [ ]:
stream = agent.stream_events(input, version="v3")

for message in stream.messages:
    for delta in message.reasoning:
        print(f"[thinking] {delta}", end="", flush=True)

    for delta in message.text:
        print(delta, end="", flush=True)

## Tool calls
- There are two useful tool-call projections:
    - message.tool_calls streams tool-call argument chunks while the model is producing the tool call.
    - stream.tool_calls streams the lifecycle of tool execution after the tool call starts.


In [ ]:
stream = agent.stream_events(input, version="v3")

for message in stream.messages:
    for chunk in message.tool_calls:
        print(f"tool call chunk: {chunk}")

    finalized = message.tool_calls.get()
    if finalized:
        print(f"finalized tool calls: {finalized}")

for call in stream.tool_calls:
    print(f"{call.tool_name}({call.input})")
    for delta in call.output_deltas:
        print(delta, end="", flush=True)
    print(call.output, call.error)

## Streaming sub-agent

In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model


def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"


weather_agent = create_agent(
    model=init_chat_model("openai:gpt-5.5"),
    tools=[get_weather],
    name="weather_agent",
)


def call_weather(query: str) -> str:
    """Query the weather agent."""
    result = weather_agent.invoke({"messages": [{"role": "user", "content": query}]})
    return result["messages"][-1].text


supervisor = create_agent(
    model=init_chat_model("openai:gpt-5.5"),
    tools=[call_weather],
    name="supervisor",
)

stream = supervisor.stream_events(
    {"messages": [{"role": "user", "content": "What's the weather in Boston?"}]},
    version="v3",
)

for subagent in stream.subagents:
    print(f"{subagent.name}: ", end="")
    for message in subagent.messages:
        for token in message.text:
            print(token, end="", flush=True)
    print()